# ML from Scratch - 4 - Naive Bayes

## Importing

In [1]:
import importlib
import my_naive_bayes

importlib.reload(my_naive_bayes)
from my_naive_bayes import Naive_Bayes

## Synthetic Data

In [2]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

texts = [
    "win free money click now",
    "free offer click here",
    "meeting about the project report",
    "lunch after the schedule meeting",
]
labels = [1, 1, 0, 0]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts, labels)

X = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
X

,about,after,click,free,here,lunch,meeting,money,now,offer,project,report,schedule,the,win
0,0,0,1,1,0,0,0,1,1,0,0,0,0,0,1
1,0,0,1,1,1,0,0,0,0,1,0,0,0,0,0
2,1,0,0,0,0,0,1,0,0,0,1,1,0,1,0
3,0,1,0,0,0,1,1,0,0,0,0,0,1,1,0


In [3]:
model = Naive_Bayes()

model.fit(X, labels)

In [4]:
from sklearn.metrics import f1_score

texts_val = [
    "claim your free prize now for free",
    "free bonus deal for you",
    "meeting about the project tomorrow",
    "please review the report after lunch",
]
labels_val = [1, 1, 0, 0]

X_val = vectorizer.transform(texts_val)
X_val = pd.DataFrame(X_val.toarray(), columns=vectorizer.get_feature_names_out())

labels_pred = model.predict(X_val)

f1_score(labels_val, labels_pred)

1.0

## Disaster Tweets

In [5]:
import pandas as pd

train_df = pd.read_csv('../data/disaster_tweets/train.csv').set_index('id')
train_df.drop(columns=['keyword', 'location'], inplace=True)

train_df.head()

,text,target
id,,
1,Our Deeds are the Reason of this #earthquake M...,1
4,Forest fire near La Ronge Sask. Canada,1
5,All residents asked to 'shelter in place' are ...,1
6,"13,000 people receive #wildfires evacuation or...",1
7,Just got sent this photo from Ruby #Alaska as ...,1


### Preprocessing

In [6]:
target = 'target'

X = train_df.drop(columns=target)
y = train_df[target]

In [7]:
X['text'] = X['text'].str.lower().str.replace(r'[^a-z ]', '', regex=True)

X.head()

,text
id,
1,our deeds are the reason of this earthquake ma...
4,forest fire near la ronge sask canada
5,all residents asked to shelter in place are be...
6,people receive wildfires evacuation orders in...
7,just got sent this photo from ruby alaska as s...


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

X_train.shape, X_val.shape

((6090, 1), (1523, 1))

In [9]:
X_train_vectorized = vectorizer.fit_transform(X_train['text'])
X_val_vectorized = vectorizer.transform(X_val['text'])
X_full_vectorized = vectorizer.transform(X['text'])

In [10]:
X_train_vectorized = pd.DataFrame(X_train_vectorized.toarray(), columns=vectorizer.get_feature_names_out(), index=X_train.index)
X_val_vectorized = pd.DataFrame(X_val_vectorized.toarray(), columns=vectorizer.get_feature_names_out(), index=X_val.index)
X_full_vectorized = pd.DataFrame(X_full_vectorized.toarray(), columns=vectorizer.get_feature_names_out(), index=X.index)

### Model Training

#### My Model

In [11]:
model_nb = Naive_Bayes()
model_nb.fit(X_train_vectorized, y_train)

In [12]:
y_val_pred_nb = model_nb.predict(X_val_vectorized)

f1_score(y_val, y_val_pred_nb)

0.7738193869096934

#### SciKit-Learn Model

In [13]:
from sklearn.naive_bayes import MultinomialNB

model_sklearn = MultinomialNB()

model_sklearn.fit(X_train_vectorized, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [14]:
y_val_pred_sklearn = model_sklearn.predict(X_val_vectorized)

f1_score(y_val, y_val_pred_sklearn)

0.7738193869096934

### Retraining and Submission

In [15]:
model_nb.fit(X_full_vectorized, y)
model_sklearn.fit(X_full_vectorized, y)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [16]:
test_df = pd.read_csv('../data/disaster_tweets/test.csv').set_index('id')
test_df.drop(columns=['keyword', 'location'], inplace=True)

test_df.head()

,text
id,
0,Just happened a terrible car crash
2,"Heard about #earthquake is different cities, s..."
3,"there is a forest fire at spot pond, geese are..."
9,Apocalypse lighting. #Spokane #wildfires
11,Typhoon Soudelor kills 28 in China and Taiwan


In [17]:
test_df['text'] = test_df['text'].str.lower().str.replace(r'[^a-z ]', '', regex=True)

test_df.head()

,text
id,
0,just happened a terrible car crash
2,heard about earthquake is different cities sta...
3,there is a forest fire at spot pond geese are ...
9,apocalypse lighting spokane wildfires
11,typhoon soudelor kills in china and taiwan


In [18]:
X_test_vectorized = vectorizer.transform(test_df['text'])
X_test_vectorized = pd.DataFrame(X_test_vectorized.toarray(), columns=vectorizer.get_feature_names_out(), index=test_df.index)

X_test_vectorized.head()

,aa,aaaaaaallll,aaarrrgghhhhttptcoguimbekge,aaceorg,aal,aampb,aampw,aan,aannnnd,aar,...,zone,zonesthank,zoom,zouma,zourryart,zrnf,zss,zumiez,zurich,zxathetis
id,,,,,,,,,,,,,,,,,,,,,
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
11,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [19]:
y_pred_nb = model_nb.predict(X_test_vectorized)

submission_df_nb = pd.DataFrame(y_pred_nb, index=test_df.index, columns=['target'])

submission_df_nb.to_csv('../data/disaster_tweets/submissions/submission_nb_my_model.csv', sep=',')

In [20]:
y_pred_sklearn = model_sklearn.predict(X_test_vectorized)

submission_df_sklearn = pd.DataFrame(y_pred_sklearn, index=test_df.index, columns=['target'])

submission_df_sklearn.to_csv('../data/disaster_tweets/submissions/submission_nb_sklearn.csv', sep=',')

Kaggle Score for my model: `0.79742`

Kaggle Score for SciKit-Learn model: `0.79742`